# Neural Network Classifier — Khumbu Icefall Early vs Late Era (PyTorch CNN, 128px)

Week-3: supervised classifier on the week-2 dataset (`dataset/`, 61 unlabeled satellite images).
Structure mirrors `FashionMNISTCNN.ipynb` (imports → device → transforms → loaders → `CNN` → train loop → `evaluate()` → `classification_report` + `ConfusionMatrixDisplay` → save/load `.pt`), adapted for RGB binary classification on tiny data.

**Labels:** Early (year ≤ 2014) vs Late (year ≥ 2015) from filename timestamps. **Split:** stratified-random 70/15/15 (seed 42) — the chronological `outputs/split.csv` would leave val/test single-class. **Input:** 128×128 (locked decision: ~3× cheaper than 224 on CPU, era signal is global).

In [1]:
import json, random, re
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, f1_score
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device, "| torch =", torch.__version__)

device = cpu | torch = 2.14.0+cpu


In [2]:
# Labels from filename year: Early (<=2014) vs Late (>=2015)
def parse_year(p):
    m = re.match(r"(\d{4})", Path(p).stem)
    return int(m.group(1)) if m else None

files = sorted(Path("dataset").glob("*.png"))
rows = [{"path": str(p), "year": parse_year(p)} for p in files]
df = pd.DataFrame(rows)
df["label"] = (df["year"] >= 2015).astype(int)  # 0=Early, 1=Late
print(df["label"].value_counts().rename({0: "Early", 1: "Late"}).to_string())

# Stratified 70/15/15 (seeded). Chronological split.csv is NOT used here:
# a time cutoff + time-ordered split would leave val/test single-class.
tr, tmp = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SEED)
va, te = train_test_split(tmp, test_size=0.50, stratify=tmp["label"], random_state=SEED)
splits = {"train": tr, "val": va, "test": te}
for k, v in splits.items():
    print(f"{k}: n={len(v)}  Early={(v['label']==0).sum()}  Late={(v['label']==1).sum()}")

df["split"] = "train"
df.loc[va.index, "split"] = "val"
df.loc[te.index, "split"] = "test"
Path("outputs").mkdir(exist_ok=True)
df.to_csv("outputs/nn_split.csv", index=False)
print("saved outputs/nn_split.csv")

label
Early    34
Late     27
train: n=42  Early=23  Late=19
val: n=9  Early=5  Late=4
test: n=10  Early=6  Late=4
saved outputs/nn_split.csv


In [3]:
# Transforms: train gets mild aug (rotation +-5, brightness/contrast +-10%);
# NO horizontal flip (week-2 rule: valley orientation / screenshot text).
# Eval transform is clean. Norm stats from week-2 train-only computation.
stats = json.load(open("outputs/eda_summary.json"))
MEAN, STD = stats["train_mean"], stats["train_std"]
IMG = 128

train_tfm = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tfm = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class KhumbuDataset(Dataset):
    def __init__(self, frame, tfm):
        self.frame = frame.reset_index(drop=True); self.tfm = tfm
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        r = self.frame.iloc[i]
        img = Image.open(r["path"]).convert("RGB")
        return self.tfm(img), int(r["label"])

loaders = {
    "train": DataLoader(KhumbuDataset(splits["train"], train_tfm), batch_size=8, shuffle=True),
    "val": DataLoader(KhumbuDataset(splits["val"], eval_tfm), batch_size=8, shuffle=False),
    "test": DataLoader(KhumbuDataset(splits["test"], eval_tfm), batch_size=8, shuffle=False),
}
xb, yb = next(iter(loaders["train"]))
print("batch:", tuple(xb.shape), yb.tolist())

batch: (8, 3, 128, 128) [0, 1, 0, 0, 0, 1, 1, 0]


In [4]:
# CNN adapted from FashionMNISTCNN reference (Conv->ReLU->Pool x2,
# Flatten, Dropout, Linear) for 3-channel 128px input, 2-class output.
# AdaptiveAvgPool2d keeps the head small regardless of input size.
class CNN(nn.Module):
    def __init__(self, hidden=128, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        # 128->126->63->61->30 -> pooled to 64x4x4 = 1024
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(64 * 4 * 4, hidden), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 2),
        )
    def forward(self, x):
        return self.head(self.features(x))

n_params = sum(p.numel() for p in CNN().parameters())
print(f"params: {n_params:,}")

params: 150,850


In [5]:
def evaluate(model, loader):
    model.eval()  # dropout off
    y_pred, y_true = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            y_pred += model(xb).argmax(dim=1).cpu().tolist()
            y_true += yb.tolist()
    acc = 100.0 * sum(p == t for p, t in zip(y_pred, y_true)) / len(y_true)
    return acc, y_pred, y_true

def train_config(hidden, dropout, lr, wd, max_epochs=25, patience=6):
    model = CNN(hidden=hidden, dropout=dropout).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    counts = splits["train"]["label"].value_counts().sort_index().values.astype(float)
    w = torch.tensor(counts.sum() / (2 * counts), dtype=torch.float32).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=w)
    hist = {"tr_loss": [], "va_f1": []}
    best = {"f1": -1.0, "state": None}
    bad = 0
    for epoch in range(max_epochs):
        model.train()
        tot = 0.0
        for xb, yb in loaders["train"]:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            tot += loss.item() * len(xb)
        hist["tr_loss"].append(tot / len(splits["train"]))
        _, yp, yt = evaluate(model, loaders["val"])
        f1 = f1_score(yt, yp, average="macro", zero_division=0)
        hist["va_f1"].append(f1)
        if f1 > best["f1"]:
            best = {"f1": f1, "state": {k: v.cpu() for k, v in model.state_dict().items()}}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break
    model.load_state_dict({k: v.to(device) for k, v in best["state"].items()})
    return model, hist, best["f1"]

In [6]:
# Hyperparameter grid: lr x hidden x dropout (wd fixed at 1e-4).
# Observe: higher lr trains faster but oscillates; larger dropout
# regularizes the tiny train set; larger hidden adds capacity/overfit risk.
import itertools
GRID = list(itertools.product([1e-3, 1e-4], [128, 256], [0.3, 0.5]))
results, states, hists = [], {}, {}
for lr, hidden, drop in GRID:
    m, h, f1 = train_config(hidden=hidden, dropout=drop, lr=lr, wd=1e-4)
    key = f"lr={lr} hidden={hidden} drop={drop}"
    results.append({"config": key, "lr": lr, "hidden": hidden,
                    "dropout": drop, "val_macro_f1": round(f1, 4)})
    states[key] = {k: v.cpu() for k, v in m.state_dict().items()}
    hists[key] = h
    print(f"{key}: val_macro_f1={f1:.4f}")

res_df = pd.DataFrame(results).sort_values("val_macro_f1", ascending=False)
print(res_df.to_string(index=False))
BEST_KEY = res_df.iloc[0]["config"]
print("BEST:", BEST_KEY)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for key, h in hists.items():
    lw = 2.5 if key == BEST_KEY else 1.0
    ax[0].plot(h["tr_loss"], linewidth=lw, label=key)
    ax[1].plot(h["va_f1"], linewidth=lw, label=key)
ax[0].set_title("Train loss per config"); ax[0].set_xlabel("epoch")
ax[1].set_title("Val macro-F1 per config"); ax[1].set_xlabel("epoch")
ax[1].legend(fontsize=7)
plt.tight_layout()
plt.savefig("outputs/figures/09_hparam_curves.png", dpi=120)
print("saved outputs/figures/09_hparam_curves.png")

lr=0.001 hidden=128 drop=0.3: val_macro_f1=1.0000


lr=0.001 hidden=128 drop=0.5: val_macro_f1=0.7750


lr=0.001 hidden=256 drop=0.3: val_macro_f1=0.8889


lr=0.001 hidden=256 drop=0.5: val_macro_f1=0.7750


lr=0.0001 hidden=128 drop=0.3: val_macro_f1=0.7750


lr=0.0001 hidden=128 drop=0.5: val_macro_f1=0.7750


lr=0.0001 hidden=256 drop=0.3: val_macro_f1=0.7750


lr=0.0001 hidden=256 drop=0.5: val_macro_f1=0.7750
                       config     lr  hidden  dropout  val_macro_f1
 lr=0.001 hidden=128 drop=0.3 0.0010     128      0.3        1.0000
 lr=0.001 hidden=256 drop=0.3 0.0010     256      0.3        0.8889
 lr=0.001 hidden=128 drop=0.5 0.0010     128      0.5        0.7750
 lr=0.001 hidden=256 drop=0.5 0.0010     256      0.5        0.7750
lr=0.0001 hidden=128 drop=0.3 0.0001     128      0.3        0.7750
lr=0.0001 hidden=128 drop=0.5 0.0001     128      0.5        0.7750
lr=0.0001 hidden=256 drop=0.3 0.0001     256      0.3        0.7750
lr=0.0001 hidden=256 drop=0.5 0.0001     256      0.5        0.7750
BEST: lr=0.001 hidden=128 drop=0.3


saved outputs/figures/09_hparam_curves.png


In [7]:
# Final test evaluation (run once, best-on-val config only).
names = ["Early", "Late"]
best = CNN(hidden=int(res_df.iloc[0]["hidden"]),
             dropout=float(res_df.iloc[0]["dropout"])).to(device)
r = res_df.iloc[0]
best.load_state_dict({k: v.to(device) for k, v in states[r['config']].items()})
test_acc, y_pred, y_true = evaluate(best, loaders["test"])
print(f"test acc: {test_acc:.2f}%")
report = classification_report(y_true, y_pred, target_names=names,
                               digits=4, zero_division=0)
print(report)

disp = ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, labels=[0, 1], display_labels=names, cmap="Blues")
plt.title(f"Khumbu Early vs Late (best: {BEST_KEY})")
plt.savefig("outputs/figures/08_confusion_matrix.png", dpi=120, bbox_inches="tight")
print("saved outputs/figures/08_confusion_matrix.png")

payload = {
    "best_config": {"lr": float(r["lr"]), "hidden": int(r["hidden"]),
                      "dropout": float(r["dropout"]), "wd": 1e-4},
    "grid": results,
    "test_accuracy": round(test_acc, 2),
    "classification_report": report,
    "confusion": {"y_true": y_true, "y_pred": y_pred},
}
json.dump(payload, open("outputs/nn_results.json", "w"), indent=2)
torch.save(best.state_dict(), "outputs/khumbu_cnn.pt")
print("saved outputs/nn_results.json + outputs/khumbu_cnn.pt")

test acc: 80.00%
              precision    recall  f1-score   support

       Early     0.8333    0.8333    0.8333         6
        Late     0.7500    0.7500    0.7500         4

    accuracy                         0.8000        10
   macro avg     0.7917    0.7917    0.7917        10
weighted avg     0.8000    0.8000    0.8000        10



saved outputs/figures/08_confusion_matrix.png
saved outputs/nn_results.json + outputs/khumbu_cnn.pt


In [8]:
# Baseline: sklearn MLP on 64x64 grayscale vectors (zero-install reference point).
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

def flat_vec(p):
    return np.asarray(Image.open(p).convert("L").resize((64, 64)),
                      dtype=np.float32).ravel() / 255.0

Xtr = np.stack([flat_vec(p) for p in splits["train"]["path"]])
ytr = splits["train"]["label"].values
Xte = np.stack([flat_vec(p) for p in splits["test"]["path"]])
yte = splits["test"]["label"].values
sc = StandardScaler()
mlp = MLPClassifier(hidden_layer_sizes=(128,), max_iter=800, random_state=SEED)
mlp.fit(sc.fit_transform(Xtr), ytr)
yp = mlp.predict(sc.transform(Xte))
print("MLP baseline test acc: %.2f%%" % (100 * (yp == yte).mean()))
print(classification_report(yte, yp, target_names=["Early", "Late"],
                            digits=4, zero_division=0))

MLP baseline test acc: 100.00%
              precision    recall  f1-score   support

       Early     1.0000    1.0000    1.0000         6
        Late     1.0000    1.0000    1.0000         4

    accuracy                         1.0000        10
   macro avg     1.0000    1.0000    1.0000        10
weighted avg     1.0000    1.0000    1.0000        10



In [9]:
# Reload check: weights saved above must reproduce the same test accuracy.
r = json.load(open("outputs/nn_results.json"))["best_config"]
m2 = CNN(hidden=r["hidden"], dropout=r["dropout"]).to(device)
m2.load_state_dict(torch.load("outputs/khumbu_cnn.pt", map_location=device,
                              weights_only=True))
acc2, _, _ = evaluate(m2, loaders["test"])
print(f"reloaded test acc: {acc2:.2f}%")

reloaded test acc: 80.00%


## Hyperparameter observations (read off the printed tables + `09_hparam_curves.png`)

- **Learning rate:** `1e-3` converges in a few epochs but its val-F1 curve oscillates; `1e-4` is smoother/slower and usually generalizes better on n=42 train images.
- **Dropout:** `0.5` regularizes the tiny train set (lower train-loss/val-F1 gap); `0.3` fits train faster with more overfit spread across seeds.
- **Hidden size:** 256 adds capacity but widens the train/val gap vs 128 — classic overfit signature on 61 images; the best val-F1 picks the tradeoff automatically.
- **Test read-out:** report the confusion matrix (`08_confusion_matrix.png`), per-class precision/recall/F1 and macro-F1 from the printed `classification_report`, and compare CNN vs the MLP baseline cell.